# 00 - Parallel Colab runner (reduced budgets)

Run this **same notebook in several Colab sessions at once**, one `SESSION` value per tab
(File > Open in a new tab, or open it again from GitHub). Every session clones branch `v2`,
writes checkpoints to Google Drive (`MyDrive/interceptor_v2/runs`, so a disconnected session
resumes where it stopped when re-run) and copies its result tables/figures to
`MyDrive/interceptor_v2/results_<SESSION>`.

Budgets and run specs: `scripts/run_reduced_budget.py` (same runs as notebooks 02-05, fewer steps;
the budgets are recorded in `results/tables/reduced_budget_run.json`).

| Order | SESSION | Work | Needs |
|---|---|---|---|
| 1 | `G_QUAD` | 6 guidance runs, quadcopter A3/A4 x 3 seeds | - |
| 2 | `G_FW` | 6 guidance runs, fixed wing | - |
| 3 | `G_MISSILE` | 6 guidance runs, missile | - |
| 4 | `FC_P6` | 3 flight-controller runs + 18 algorithm-selection runs | - |
| 5 | `CLASSICAL` | sim figures, A1/A2 tables, miss distributions, B1/B2 FC table | - |
| 6 | `ABL_1`, `ABL_2`, `ABL_3` | 57 ablation runs in 3 shards | - |
| 7 | `FINAL_QUAD` | A3/A4 quad table, FC with RL, closed loop, bridge sweep | G_QUAD, FC_P6 done |
| 7 | `FINAL_FW_MS` | A3/A4 fixed-wing and missile tables | G_FW, G_MISSILE done |
| 7 | `FINAL_SWEEPS` | robustness sweeps, residual analysis, videos, manifest | all G_* and FC_P6 done |

The runtime type can stay **CPU** (the simulator is the bottleneck, not the network).
A runtime with more CPU cores runs more jobs at once automatically.


In [ ]:
SESSION = "G_QUAD"   # <- change per tab: G_QUAD G_FW G_MISSILE FC_P6 CLASSICAL ABL_1 ABL_2 ABL_3 FINAL_QUAD FINAL_FW_MS FINAL_SWEEPS
BRANCH = "v2"
REPO = "https://github.com/siddhmehta5131/interceptor-drone-rl.git"

In [ ]:
import os, subprocess, sys, shutil, time
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/interceptor_v2'); (DRIVE / 'runs').mkdir(parents=True, exist_ok=True)
ROOT = Path('/content/interceptor')
if ROOT.exists():
    subprocess.run(['git', '-C', str(ROOT), 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, REPO, str(ROOT)], check=True)
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
if not (ROOT / 'runs').exists():
    os.symlink(DRIVE / 'runs', ROOT / 'runs')       # checkpoints live on Drive
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
print('CPU cores:', os.cpu_count())

In [ ]:
!pip -q install stable-baselines3==2.9.0 gymnasium filterpy scipy pandas matplotlib pyyaml
if SESSION == 'FINAL_SWEEPS':
    !apt-get -qq install -y xvfb libgl1-mesa-glx > /dev/null
    !pip -q install pyvista imageio imageio-ffmpeg trimesh
if SESSION == 'CLASSICAL':
    !pip -q install gym-pybullet-drones || echo "pybullet install failed; B3 stays pending"

In [ ]:
W = os.cpu_count() or 2
P = lambda arm, pl: f'{arm}_{pl}=runs/p7_{pl}_{arm}_s0/checkpoint.zip'
A3 = ' '.join(f'--policy {pl}=runs/p7_{pl}_a3_s0/checkpoint.zip' for pl in ('quadcopter', 'fixed_wing', 'missile'))
R = 'python scripts/run_reduced_budget.py'
JOBS = {
  'G_QUAD':      [f'{R} --phase guidance --only quadcopter --workers {W}'],
  'G_FW':        [f'{R} --phase guidance --only fixed_wing --workers {W}'],
  'G_MISSILE':   [f'{R} --phase guidance --only missile --workers {W}'],
  'FC_P6':       [f'{R} --phase fc p6 --workers {W}'],
  'ABL_1':       [f'{R} --phase ablations --shard 1/3 --workers {W}'],
  'ABL_2':       [f'{R} --phase ablations --shard 2/3 --workers {W}'],
  'ABL_3':       [f'{R} --phase ablations --shard 3/3 --workers {W}'],
  'CLASSICAL':   ['python scripts/make_sim_figures.py',
                  'python scripts/benchmark_sim.py --seconds 2',
                  f'python scripts/evaluate_guidance.py --episodes 100 --workers {W}',
                  f'python scripts/collect_guidance_distributions.py --platforms quadcopter fixed_wing missile --workers {W}',
                  'python scripts/plot_guidance.py',
                  'python scripts/evaluate_fc.py'],
  'FINAL_QUAD':  [f'python scripts/evaluate_guidance_arms.py --episodes 100 --workers {W} --policy {P("a3","quadcopter")} --policy {P("a4","quadcopter")}',
                  'python scripts/evaluate_fc.py --rl runs/p7_fc_asym_s0/checkpoint.zip',
                  f'python scripts/evaluate_closed_loop.py --episodes 100 --workers {W} --guidance-policy runs/p7_quadcopter_a3_s0/checkpoint.zip',
                  f'python scripts/sweep_bridge_rate.py --episodes 5 --workers {W} --guidance-policy runs/p7_quadcopter_a3_s0/checkpoint.zip'],
  'FINAL_FW_MS': [f'python scripts/evaluate_guidance_arms.py --episodes 100 --workers {W} --policy {P("a3","fixed_wing")} --policy {P("a4","fixed_wing")} --policy {P("a3","missile")} --policy {P("a4","missile")}'],
  'FINAL_SWEEPS': ['python scripts/build_manifest.py --runs runs --output results/manifest.json --verify',
                   f'python scripts/sweep_robustness.py --episodes 5 --workers {W} {A3}',
                   f'python scripts/residual_analysis.py {A3}',
                   f'python scripts/render_videos.py --out results/videos --resolution 1280x720 {A3} --fc-policy runs/p7_fc_asym_s0/checkpoint.zip'],
}
if SESSION == 'FINAL_SWEEPS':
    import pyvista as pv; pv.start_xvfb()
out = DRIVE / f'results_{SESSION}'
def save():
    shutil.copytree(ROOT / 'results', out, dirs_exist_ok=True)
    shutil.copytree(ROOT / 'docs' / 'figures', out / 'docs_figures', dirs_exist_ok=True)
t0 = time.time()
for cmd in JOBS[SESSION]:
    print(f'\n===== {time.strftime("%H:%M:%S")} {cmd}', flush=True)
    r = subprocess.run(cmd, shell=True)
    print('exit code', r.returncode, flush=True)
    save()                                        # results reach Drive after every step
print(f'SESSION {SESSION} finished in {(time.time()-t0)/60:.1f} min; results in {out}')

In [ ]:
# Progress of every training run on Drive (any session can run this cell)
import json, glob
done, going = 0, []
for d in sorted(glob.glob(str(DRIVE / 'runs' / '*'))):
    if os.path.exists(f'{d}/DONE'):
        done += 1
    elif os.path.exists(f'{d}/curves.csv'):
        last = open(f'{d}/curves.csv').read().strip().splitlines()[-1].split(',')[0]
        going.append(f'{os.path.basename(d)} at {last}')
print(done, 'runs done'); print('\n'.join(going))